# Module 8 — Troubleshooting and common mistakes

**Who this is for:** everyone. Use it as a reference: jump to the table that matches your problem.

> **Self-check — skip this module if you can already:**
>
> - [ ] say why `x[::4]` is not a safe way to downsample
> - [ ] explain why fitting a scaler before cross-validation inflates scores
> - [ ] recognise when a notebook is running on the wrong environment
> - [ ] find out why a filter 'did nothing' or destroyed the signal
>
> If you can tick every box, go to the next module. If not, work through the parts you can't tick. Before
> jumping to a section, run the module's **setup cell** (the first code cell): later sections use variables
> defined earlier, so run the cells above a section if you get a `NameError`.

**You will learn**
- setup and environment problems
- notebook and Python pitfalls
- signal-processing mistakes, with demonstrations
- machine-learning mistakes, with demonstrations
- a routine for when you are stuck

**Where the course uses this:** every module, lab and capstone track. Many of these mistakes appear as *Rule out* questions in the labs.

*Run each cell with **Shift+Enter**, top to bottom. Nothing here needs data files; once the libraries are
installed (see the [overview](README.md#setup-first)), the notebook runs offline.*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/python_tutorial/08_troubleshooting.ipynb)
&nbsp; Part of the **Python for biosignal processing** tutorial — see the [overview](README.md) to choose which modules you need.

## 1. Setup and environment problems

| Symptom | Likely cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'wfdb'` | not installed **in the environment the notebook uses** | check `sys.executable` (Module 6); install with `%pip install wfdb` from inside the notebook, or select the right kernel |
| Installed a library but the notebook still can't import it | notebook kernel ≠ terminal environment | VS Code: pick the `.venv` kernel; Jupyter: restart the kernel after installing |
| Error deep inside a library right after an install (e.g. `wfdb` with pandas 3) | two libraries' versions don't fit together | install the pinned versions from `requirements-lock.txt`; `pip check`; pin the working version |
| `pip` says "externally managed environment" | you are installing into the system Python | create and activate a virtual environment |
| Results differ between teammates | different library versions or an unseeded step | same lock file for everyone; seed everything (Module 6) |
| Colab: "my variables are gone" | the session reset after being idle | re-run from the top; save results to files (Module 7) |
| Colab: files disappeared | the Colab disk is temporary | save to Google Drive or download before closing |
| Windows: `Activate.ps1 cannot be loaded` | PowerShell execution policy | `Set-ExecutionPolicy -Scope Process RemoteSigned`, or use cmd |
| Out of memory | loading all subjects at once, or float64 copies of huge arrays | process one subject at a time; cache features (Module 7); delete arrays you no longer need |

## 2. Notebook and Python pitfalls

In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
rng = np.random.default_rng(2013)

# Pitfall: a slice of a NumPy array is a VIEW. Changing it changes the original.
x = np.arange(6.0)
first = x[:3]
first[0] = 99
print("original changed:", x)
safe = x[:3].copy()                       # use .copy() when you want an independent array

# Pitfall: comparing floats with ==
print(0.1 + 0.2 == 0.3, np.isclose(0.1 + 0.2, 0.3))

# Pitfall: integer overflow in raw ADC data
adc = np.array([30000, 20000], dtype=np.int16)
print("int16 sum:", adc.sum(dtype=np.int16), " as float:", adc.astype(float).sum())

| Symptom | Likely cause | Fix |
|---|---|---|
| Works after running cells in some order, fails on "Run all" | hidden state from cells run out of order | restart and run all, regularly |
| A variable has an "impossible" value | it was overwritten in a later cell, or is a view of another array | search for where it is assigned; use `.copy()` |
| `IndexError` on the last epoch | the recording is not a whole number of epochs | trim: `x[: n_ep * w]` |
| `ValueError: operands could not be broadcast` | shapes don't match, e.g. `(n_samples, n_ch)` vs `(n_ch, n_samples)` | print `.shape` of both; transpose with `.T` |
| Plot shows a flat line or nonsense time axis | plotting against sample index, or wrong `fs` | build `t = np.arange(n) / fs`; check `fs` from the file |
| A function silently returns `None` | missing `return` | add `return`; test with an `assert` |

## 3. Signal-processing mistakes

**Mistake: downsampling by skipping samples without filtering first.** Anything above the new Nyquist
frequency folds back as a fake low frequency. (Skipping samples is fine only after a low-pass filter has removed
everything above the new Nyquist frequency; `resample_poly` and `decimate` do both steps.)

In [ ]:
fs = 1000
t = np.arange(2 * fs) / fs
x = np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 180 * t)      # 5 Hz signal + 180 Hz component

bad = x[::5]                                           # 1000 -> 200 Hz by skipping: 180 Hz aliases to 20 Hz
good = signal.resample_poly(x, up=1, down=5)           # filters first, then downsamples
for name, y in [("x[::5]", bad), ("resample_poly", good)]:
    f, P = signal.welch(y, fs=200, nperseg=200)
    print(f"{name:14s} power at 20 Hz: {P[np.argmin(abs(f - 20))]:.4f}")

**Mistake: designing a filter with the wrong sampling rate.** If you forget `fs=`, scipy assumes cut-offs
are fractions of Nyquist, and the filter lands somewhere else entirely.

In [ ]:
fs = 250
print("correct:", signal.butter(4, 40, btype="lowpass", fs=fs, output="sos").shape)
try:
    signal.butter(4, 40, btype="lowpass", output="sos")        # no fs: 40 is read as 40 x Nyquist
except ValueError as e:
    print("without fs:", e)

**Mistake: filtering short epochs one by one.** Every filter needs time to settle; on short pieces the
edges are distorted. Filter the whole recording first, then cut it into epochs.

In [ ]:
fs = 100
t = np.arange(60 * fs) / fs
x = np.sin(2 * np.pi * 1.0 * t)
sos = signal.butter(4, 0.5, btype="highpass", fs=fs, output="sos")

whole_then_cut = signal.sosfiltfilt(sos, x)[: 2 * fs]
cut_then_filter = signal.sosfiltfilt(sos, x[: 2 * fs])
print(f"max error vs filtering the whole record: {np.abs(cut_then_filter - whole_then_cut).max():.3f}")

| Mistake | What goes wrong | Do instead |
|---|---|---|
| Frequency axis built by hand | peaks at the wrong frequency | `np.fft.rfftfreq(n, d=1/fs)`; `welch(..., fs=fs)` |
| Raw periodogram of a short, noisy record | random peaks look like rhythms | Welch / multitaper; check a peak persists (Lab 7) |
| High-pass corner too high | slow content you need (delta, ST segment) disappears | check what must survive (Labs 8–9) |
| `filtfilt` in a real-time setting | uses future samples | causal `sosfilt`, and account for the delay |
| `b, a` form at high order | unstable or wrong output | `output="sos"` with `sosfiltfilt` |
| Notch at the wrong mains frequency | 50 Hz in Europe, 60 Hz in the US | check the spectrum first |
| Mixing units (V vs µV) | thresholds off by a factor of a million | write units in variable names (`eeg_uv`) and axis labels |
| Hilbert envelope / frequency on a broadband signal | meaningless numbers | band-pass to one component first (Lab 5) |

## 4. Machine-learning mistakes

**Mistake: anything that learns from data, fitted before the split.** Here a feature *selector* sees all the
labels before cross-validation. On pure noise, it still "finds" good features, and the score looks real.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

X = rng.standard_normal((100, 2000))                 # pure noise features
y = rng.integers(0, 2, 100)                          # random labels: the true accuracy is 50 %
cv = StratifiedKFold(5, shuffle=True, random_state=2013)

X_sel = SelectKBest(f_classif, k=20).fit_transform(X, y)              # WRONG: selection saw every label
leaky = cross_val_score(LogisticRegression(max_iter=1000), X_sel, y, cv=cv).mean()
honest = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)),
                         X, y, cv=cv).mean()                          # RIGHT: selection inside each fold
print(f"selection before CV: {leaky:.2f}   <- looks like skill, it isn't")
print(f"selection inside CV: {honest:.2f}   <- about chance, the truth")

| Mistake | What goes wrong | Do instead |
|---|---|---|
| Random split when the question is "new subject" | epochs of one person in train and test: inflated scores | `GroupKFold` / your track's split unit (Module 5) |
| Scaler, selector, SMOTE or tuning fitted on all data | test information leaks in | put them inside a `Pipeline` (for SMOTE: `imblearn.pipeline.Pipeline`); tune with nested CV |
| Accuracy only, on imbalanced classes | "always the majority class" looks good | confusion matrix, kappa, macro-F1 |
| One score, no spread | a lucky fold looks like a result | mean ± std across folds |
| Comparing models on different splits | the comparison is unfair | same folds for every model |
| Tuning on the hold-out set | the hold-out is no longer unseen | tune in cross-validation; touch the hold-out once |
| No baseline | can't tell if a model learned anything | always report a trivial baseline and the supplied one |
| Unseeded randomness | numbers change every run | `random_state=` everywhere (Module 6) |

## 5. When you are stuck

1. **Restart and run all.** Does the problem survive a clean run?
2. **Read the last line of the error,** then find the first line in the traceback that is *your* code.
3. **Shrink it:** one subject, one epoch, ten samples. Print `shape`, `dtype` and a few values.
4. **Plot the signal** before and after the step that looks wrong.
5. **Compare with a known answer:** a synthetic sine at a known frequency, a known number of beats.
6. **Check the environment:** `sys.executable` and library versions (Module 6).
7. **Ask** in the *Capstone Q&A* discussion with the full error message, the smallest code that shows it,
   what you tried, and your track. A good question often answers itself while you write it.

**Next:** put this into practice in Module 9, the [debugging challenge](09_debugging_challenge.ipynb): ten
broken cells, one bug each, for you to find and fix.